# Customer Churn Prediction Using Machine Learning

## 1. Introduction
This project predicts customer churn using ML. Customer churn refers to customers leaving a company. Predicting churn helps businesses take proactive measures to retain customers.

## 2. Problem Statement
Given customer data (demographics, account info, services), predict whether a customer will churn.

## 3. Objective
Build and compare ML models to predict customer churn, identify key factors contributing to churn.

## 4. Dataset Description
Using IBM Telco Customer Churn dataset with 7043 customer records and 21 features. Describe key features.

## 5. Importing Libraries

In [ ]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('All libraries imported successfully!')

## 6. Loading Dataset

In [ ]:
# Load the dataset (handles both local Jupyter and Google Colab)
import os

# Automatic path resolution:
file_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
if not os.path.exists(file_path):
    if os.path.exists('data/WA_Fn-UseC_-Telco-Customer-Churn.csv'):
        file_path = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
    elif os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
        file_path = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
    else:
        # Direct URL fallback for Google Colab
        file_path = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

df = pd.read_csv(file_path)
print(f'Dataset loaded successfully from: {file_path}')
print(f'Dataset shape: {df.shape}')

## 7. Data Understanding

In [ ]:
# Display first 5 rows
print('--- First 5 Rows ---')
df.head()

In [ ]:
# Dataset shape
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

In [ ]:
# Column names
print('--- Column Names ---')
print(df.columns.tolist())

In [ ]:
# Data types
print('--- Data Types ---')
print(df.dtypes)

In [ ]:
# Missing values
print('--- Missing Values ---')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Duplicate values
print(f'Number of duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary
print('--- Statistical Summary ---')
df.describe()

In [ ]:
# Check unique values for each column
print('--- Unique Values per Column ---')
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values')

## 8. Data Cleaning and Preprocessing
In this section, we handle missing values, convert data types, remove unnecessary columns, and encode categorical variables.

In [ ]:
# Step 1: Convert TotalCharges to numeric
# TotalCharges has some blank/space values that need to be handled
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Missing values in TotalCharges after conversion: {df["TotalCharges"].isnull().sum()}')

In [ ]:
# Step 2: Handle missing values in TotalCharges
# Fill missing TotalCharges with median value
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'Missing values after filling: {df["TotalCharges"].isnull().sum()}')

In [ ]:
# Step 3: Remove customerID column (not useful for prediction)
df = df.drop('customerID', axis=1)
print('customerID column removed.')
print(f'Remaining columns: {df.shape[1]}')

In [ ]:
# Step 4: Encode the target variable (Churn)
# Yes -> 1, No -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Target variable encoded:')
print(df['Churn'].value_counts())

In [ ]:
# Step 5: Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('Churn')  # Remove target from numerical features

print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')
print(f'Numerical columns ({len(numerical_cols)}): {numerical_cols}')

In [ ]:
# Step 6: One-Hot Encoding for categorical variables
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f'Shape after encoding: {df_encoded.shape}')
print('\nEncoded columns:')
print(df_encoded.columns.tolist())

## 9. Exploratory Data Analysis
Let's visualize the data to understand patterns and relationships.

In [ ]:
# 1. Churn Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Customer Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Churn (0 = No, 1 = Yes)', fontsize=12)
plt.ylabel('Count', fontsize=12)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

print(f'\nChurn Rate: {df["Churn"].mean()*100:.2f}%')

In [ ]:
# 2. Churn vs Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(x='Contract', hue='Churn', data=df, palette='Set2')
plt.title('Churn vs Contract Type', fontsize=14, fontweight='bold')
plt.xlabel('Contract Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title='Churn', labels=['No (0)', 'Yes (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# 3. Churn vs Internet Service
plt.figure(figsize=(8, 5))
sns.countplot(x='InternetService', hue='Churn', data=df, palette='Set2')
plt.title('Churn vs Internet Service', fontsize=14, fontweight='bold')
plt.xlabel('Internet Service Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title='Churn', labels=['No (0)', 'Yes (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# 4. Churn vs Payment Method
plt.figure(figsize=(10, 5))
sns.countplot(x='PaymentMethod', hue='Churn', data=df, palette='Set2')
plt.title('Churn vs Payment Method', fontsize=14, fontweight='bold')
plt.xlabel('Payment Method', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=25, ha='right')
plt.legend(title='Churn', labels=['No (0)', 'Yes (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# 5. Churn vs Tenure
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='tenure', hue='Churn', kde=True, palette='Set2', bins=30)
plt.title('Churn vs Tenure (Months)', fontsize=14, fontweight='bold')
plt.xlabel('Tenure (Months)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Monthly Charges Distribution
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, palette='Set2', bins=30)
plt.title('Monthly Charges Distribution by Churn', fontsize=14, fontweight='bold')
plt.xlabel('Monthly Charges ($)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Correlation Heatmap for Numerical Features
plt.figure(figsize=(10, 8))
numerical_data = df_encoded.select_dtypes(include=['int64', 'float64'])
corr_matrix = numerical_data.corr()

# Show only correlations with Churn for cleaner view
churn_corr = corr_matrix['Churn'].sort_values(ascending=False)
print('--- Correlation with Churn ---')
print(churn_corr)

# Heatmap of top correlated features
top_features = churn_corr.head(10).index.tolist() + churn_corr.tail(5).index.tolist()
top_features = list(set(top_features))
plt.figure(figsize=(12, 8))
sns.heatmap(df_encoded[top_features].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap (Top Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Feature Engineering
Separate features (X) and target (y) for model training.

In [ ]:
# Separate features and target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nTarget distribution:')
print(y.value_counts())
print(f'\nChurn rate: {y.mean()*100:.2f}%')

## 11. Train-Test Split

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size: {X_test.shape[0]}')
print(f'\nTraining set churn rate: {y_train.mean()*100:.2f}%')
print(f'Testing set churn rate: {y_test.mean()*100:.2f}%')

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Feature scaling applied using StandardScaler.')
print(f'Scaled training data shape: {X_train_scaled.shape}')

## 12. Model Training
We will train three models: Logistic Regression, Decision Tree, and Random Forest.

In [ ]:
# =============================================
# Model 1: Logistic Regression
# =============================================
print('=' * 50)
print('Model 1: Logistic Regression')
print('=' * 50)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

print('Logistic Regression trained successfully!')

In [ ]:
# =============================================
# Model 2: Decision Tree Classifier
# =============================================
print('=' * 50)
print('Model 2: Decision Tree Classifier')
print('=' * 50)

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

print('Decision Tree trained successfully!')

In [ ]:
# =============================================
# Model 3: Random Forest Classifier
# =============================================
print('=' * 50)
print('Model 3: Random Forest Classifier')
print('=' * 50)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

print('Random Forest trained successfully!')

## 13. Model Evaluation

In [ ]:
# Function to evaluate a model
def evaluate_model(name, y_true, y_pred):
    """Evaluate and display model performance metrics."""
    print('=' * 50)
    print(f'Model: {name}')
    print('=' * 50)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f'Accuracy:  {accuracy*100:.2f}%')
    print(f'Precision: {precision*100:.2f}%')
    print(f'Recall:    {recall*100:.2f}%')
    print(f'F1 Score:  {f1*100:.2f}%')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['No Churn', 'Churn']))
    
    return {'Model': name, 'Accuracy': round(accuracy*100, 2), 
            'Precision': round(precision*100, 2), 
            'Recall': round(recall*100, 2), 
            'F1 Score': round(f1*100, 2)}

In [ ]:
# Evaluate all models
results = []
results.append(evaluate_model('Logistic Regression', y_test, lr_pred))

In [ ]:
results.append(evaluate_model('Decision Tree', y_test, dt_pred))

In [ ]:
results.append(evaluate_model('Random Forest', y_test, rf_pred))

## 14. Model Comparison

In [ ]:
# Create comparison table
results_df = pd.DataFrame(results)
print('\n' + '=' * 60)
print('MODEL COMPARISON TABLE')
print('=' * 60)
print(results_df.to_string(index=False))
print('=' * 60)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors = ['#66c2a5', '#fc8d62', '#8da0cb']

for i, metric in enumerate(metrics):
    bars = axes[i].bar(results_df['Model'], results_df[metric], color=colors)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylim(50, 100)
    axes[i].set_ylabel('Score (%)', fontsize=11)
    axes[i].tick_params(axis='x', rotation=30)
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[i].annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                        xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 15. Best Model

In [ ]:
# Select the best model based on F1 Score
best_model_row = results_df.loc[results_df['F1 Score'].idxmax()]
best_model_name = best_model_row['Model']

print('=' * 50)
print('BEST MODEL SELECTION')
print('=' * 50)
print(f'Best Model: {best_model_name}')
print(f'Accuracy:   {best_model_row["Accuracy"]}%')
print(f'Precision:  {best_model_row["Precision"]}%')
print(f'Recall:     {best_model_row["Recall"]}%')
print(f'F1 Score:   {best_model_row["F1 Score"]}%')
print('=' * 50)
print(f'\nThe {best_model_name} model is selected as the best model')
print('because it achieved the highest F1 Score, which balances')
print('both precision and recall - important for churn prediction')
print('where we need to identify churning customers (recall) while')
print('also being accurate in our predictions (precision).')

In [ ]:
# Confusion Matrix for the Best Model
# Map model name to predictions
model_preds = {
    'Logistic Regression': lr_pred,
    'Decision Tree': dt_pred,
    'Random Forest': rf_pred
}

best_pred = model_preds[best_model_name]
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'], linewidths=1, linecolor='black')
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nTrue Negatives (correctly predicted No Churn): {cm[0][0]}')
print(f'False Positives (incorrectly predicted Churn): {cm[0][1]}')
print(f'False Negatives (missed Churn customers): {cm[1][0]}')
print(f'True Positives (correctly predicted Churn): {cm[1][1]}')

## 16. Feature Importance

In [ ]:
# Feature Importance (using Random Forest)
# Random Forest provides built-in feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('--- Top 10 Most Important Features ---')
print(feature_importance.head(10).to_string(index=False))

# Visualize top 10 features
plt.figure(figsize=(10, 6))
top_10 = feature_importance.head(10)
sns.barplot(x='Importance', y='Feature', data=top_10, palette='viridis')
plt.title('Top 10 Most Important Features for Churn Prediction', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

print('\n--- What These Features Mean ---')
print('1. TotalCharges / MonthlyCharges: Higher charges may lead to dissatisfaction.')
print('2. tenure: Newer customers are more likely to churn.')
print('3. Contract type: Month-to-month contracts have higher churn rates.')
print('4. Internet Service: Fiber optic users tend to churn more (possibly due to cost).')
print('5. Payment Method: Electronic check users show higher churn rates.')
print('6. Online Security / Tech Support: Lack of these services increases churn risk.')

## 17. Customer Churn Prediction

In [ ]:
# Save the model objects for prediction
model_objects = {
    'Logistic Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model
}
best_model = model_objects[best_model_name]

def predict_customer_churn(customer_data):
    """
    Predict whether a customer is likely to churn.
    
    Parameters:
    -----------
    customer_data : dict
        Dictionary containing customer information with feature names as keys.
    
    Returns:
    --------
    str : Prediction result
    """
    # Create a DataFrame from customer data
    customer_df = pd.DataFrame([customer_data])
    
    # Scale the features using the same scaler
    customer_scaled = scaler.transform(customer_df)
    
    # Make prediction
    prediction = best_model.predict(customer_scaled)
    probability = best_model.predict_proba(customer_scaled)
    
    if prediction[0] == 1:
        result = 'Customer is likely to CHURN'
        confidence = probability[0][1] * 100
    else:
        result = 'Customer is likely to STAY'
        confidence = probability[0][0] * 100
    
    print(f'Prediction: {result}')
    print(f'Confidence: {confidence:.2f}%')
    return result

In [ ]:
# Test prediction with a sample customer
# Use the first row of test set as an example
sample_customer = X_test.iloc[0]

print('--- Sample Customer Information ---')
print(f'Number of features: {len(sample_customer)}')
print(f'\nActual Churn: {"Yes" if y_test.iloc[0] == 1 else "No"}')
print()

# Make prediction
result = predict_customer_churn(sample_customer.to_dict())

In [ ]:
# Test with another sample
sample_customer_2 = X_test.iloc[5]
print('--- Another Sample Customer ---')
print(f'Actual Churn: {"Yes" if y_test.iloc[5] == 1 else "No"}')
print()
result2 = predict_customer_churn(sample_customer_2.to_dict())

## 18. Save the Model

In [ ]:
# Save the best model and artifacts
import os

model_dir = '../models'
try:
    os.makedirs(model_dir, exist_ok=True)
except Exception:
    model_dir = 'models'
    os.makedirs(model_dir, exist_ok=True)

# Save the best model
joblib.dump(best_model, os.path.join(model_dir, 'churn_model.pkl'))
print(f'Best model ({best_model_name}) saved to {model_dir}/churn_model.pkl')

# Save the scaler
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))
print(f'Scaler saved to {model_dir}/scaler.pkl')

# Save feature names for future reference
joblib.dump(X.columns.tolist(), os.path.join(model_dir, 'feature_names.pkl'))
print(f'Feature names saved to {model_dir}/feature_names.pkl')

print('\nAll model artifacts saved successfully!')

## 18. Conclusion
In this project, we built a machine learning model to predict customer churn using the IBM Telco Customer Churn dataset.

Key Findings:
- Customer churn is influenced by factors like contract type, tenure, monthly charges, and internet service.
- Month-to-month contract customers have the highest churn rate.
- Customers with fiber optic internet service tend to churn more.
- Newer customers (low tenure) are more likely to churn.

In [ ]:
# ==========================================
# FINAL RESULTS
# ==========================================
print('=' * 50)
print('FINAL RESULTS')
print('=' * 50)
print(f'Best Model: {best_model_name}')
print(f'Accuracy: {best_model_row["Accuracy"]}%')
print(f'Precision: {best_model_row["Precision"]}%')
print(f'Recall: {best_model_row["Recall"]}%')
print(f'F1 Score: {best_model_row["F1 Score"]}%')
print('=' * 50)
print('\nProject completed successfully!')

## 19. Future Scope
- **More Algorithms**: Try Gradient Boosting, XGBoost, or SVM for potentially better results.
- **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV to optimize model parameters.
- **Feature Engineering**: Create new features like charge-to-tenure ratio.
- **Real-time Prediction**: Build an API to make real-time churn predictions.
- **Customer Segmentation**: Use clustering to group customers by behavior.
- **Deep Learning**: Explore neural networks for complex pattern recognition.